In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc

import numpy as np
import pandas as pd

from transformers import pipeline
from transformers import AutoTokenizer, AutoConfig
from transformers import DataCollatorForLanguageModeling, GPT2LMHeadModel,  GPT2Tokenizer, GPT2Config
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

import tensorflow as tf
from transformers import create_optimizer, AdamWeightDecay
from transformers import TFAutoModelForCausalLM

import evaluate
import datasets
from datasets import ClassLabel, load_dataset, Dataset, DatasetDict, load_metric

print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

In [ ]:
class CFG:
    wandb = False
    report_to = None
    lab_assignment = 5
    _wandb_kernel = "temuujin"

    debug = False
    num_workers = 12

    tokenizer_name = 'bayartsogt/mongolian-gpt2'
    model_name = 'bayartsogt/mongolian-gpt2'

    project = 'NUM-Machine-Learning-Lab-5'
    name = "Lab 5"

    config = {
        "output_dir": "lab5_finetune_gpt2",
        "group": model_name,
        "learning_rate": 2e-5,
        "weight_decay": 1e-2,
        'num_train_epochs': 3,
        "train_batch_size": 32,
        "eval_batch_size": 32,
        "dataloader_num_workers": num_workers,
        "finetuning_task": 'ner',
        "evaluation_strategy": 'epoch',
        "logging_strategy": 'epoch',
        "overwrite_output_dir": True,
        "push_to_hub": False,
    }

    model_save_dir = "lab5_gpt2"

    test_size = 0.2

    train = True
    eval = True

    eval_metric = "seqeval"

    early_stopping_patience = 15

if CFG.debug:
    CFG.config['num_train_epochs'] = 2

if CFG.wandb:
    os.environ["WANDB_SILENT"] = "True"
    CFG.report_to = "wandb"

    import wandb
    wandb.login()

    run = wandb.init(
        project = CFG.project,
        name = CFG.name,
        config = CFG.config
    )

config = CFG.config

In [ ]:
def concatenate_columns(example):
    example["prompt"] = example["prompt"] + " " + example["answer"]
    return example

df = pd.read_csv('khk.noun.tsv', sep = '\t')

df['prompt'] = '<s> bb: ' + df['prompt']
df['answer'] = df['answer'] + '</s>'

infl_dataset = Dataset.from_pandas(df)
ds_train_devtest = infl_dataset.train_test_split(test_size = 0.025, seed = 42)
ds_devtest = ds_train_devtest['test'].train_test_split(test_size = 0.5, seed = 42)

ds_splits = DatasetDict({
    'train': ds_train_devtest['train'],
    'valid': ds_devtest['train'],
    'test': ds_devtest['test']
})

ds_splits["train"] = ds_splits["train"].map(concatenate_columns)
ds_splits["valid"] = ds_splits["valid"].map(concatenate_columns)

ds_splits = ds_splits.flatten()

ds_splits

In [ ]:
block_size = 64
tokenizer = AutoTokenizer.from_pretrained(CFG.tokenizer_name)

def preprocess_function(examples):
    return tokenizer(examples["prompt"])

tokenized_ds = ds_splits.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = ds_splits["train"].column_names,
)

In [ ]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

lm_dataset = tokenized_ds.map(group_texts, batched = True, num_proc = 4)

In [ ]:
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer, mlm = False, return_tensors = "tf")

model = TFAutoModelForCausalLM.from_pretrained(CFG.model_name)

tf_train_set = model.prepare_tf_dataset(
    lm_dataset["train"],
    shuffle = True,
    batch_size = 12,
    collate_fn = data_collator,
)

tf_test_set = model.prepare_tf_dataset(
    lm_dataset["test"],
    shuffle = False,
    batch_size = 12,
    collate_fn = data_collator,
)

In [ ]:
model.compile(
    optimizer = AdamWeightDecay(learning_rate = 2e-5, weight_decay_rate = 0.01),
    loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits = True),
    metrics = tf.metrics.SparseCategoricalAccuracy(),
)

In [ ]:
model.fit(x = tf_train_set, validation_data = tf_test_set, epochs = 3)